# Export Base Data

This notebook exports the base dataset (video embeddings and metadata) into specific JSON formats for downstream consumption. It provides a channel-to-titles mapping and a structured list of channels with their respective video details.

## Setup and Environment

This cell mounts Google Drive and configures the input and output paths. It supports overriding the input path via the `DATA_PATH` environment variable for testing purposes.

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab, skipping drive mount.")

DEFAULT_DATA_PATH = '/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv'
DATA_PATH = Path(os.environ.get('DATA_PATH', DEFAULT_DATA_PATH))

DEFAULT_OUTPUT_DIR = '/content/drive/MyDrive/Graphiko/exports/base_data/latest'
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', DEFAULT_OUTPUT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input path: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## Load Base Dataset

This cell loads the reduced 20-dimensional video embeddings and metadata from the specified CSV file.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data not found at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows.")
df.head()

## Format 1: Channel-to-Titles Dictionary

This cell creates a dictionary where keys are channel names and values are lists of video titles associated with each channel. The resulting JSON is exported to Google Drive.

In [ ]:
format1 = df.groupby('channel_name')['video_title'].apply(list).to_dict()

export_path1 = OUTPUT_DIR / 'channel_titles.json'
with open(export_path1, 'w') as f:
    json.dump(format1, f, indent=2)

print(f"Format 1 exported to: {export_path1}")

## Format 2: Structured Channel and Video List

This cell creates a list of maps, each containing the channel ID, channel name, and a list of videos (including title, ID, and view count). The resulting JSON is exported to Google Drive.

In [ ]:
format2 = []
for (cid, cname), group in df.groupby(['channel_id', 'channel_name']):
    videos = group[['video_title', 'video_id', 'view_count']].rename(columns={
        'video_title': 'title',
        'video_id': 'id',
        'view_count': 'views'
    }).to_dict(orient='records')
    
    format2.append({
        'channel_id': cid,
        'channel_name': cname,
        'videos': videos
    })

export_path2 = OUTPUT_DIR / 'channels_structured.json'
with open(export_path2, 'w') as f:
    json.dump(format2, f, indent=2)

print(f"Format 2 exported to: {export_path2}")